# Predicting election winners with XGBoost

Loads the long poll dataset (`polls_long_with_results.csv` from `build_dataset.ipynb`), engineers features (polls **plus** fundamentals: partisan lean, incumbency, prior margin), collapses to **one row per candidate per race**, and trains a gradient-boosted classifier for `won`.

**Honest framing up front:** for *calling the winner*, the polling leader is already right ~88–95% of the time, so a model barely beats that on accuracy. The model's value is **calibrated win probabilities** (useful for close races, ratings, betting markets) — judged by AUC / log-loss / Brier, not just accuracy.

**Validation:** split by year — train on 2018/2020/2022, test on 2024 (~133 races, each House district counted separately) — so we never leak the future.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (accuracy_score, roc_auc_score, log_loss,
                             brier_score_loss, confusion_matrix)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)
print('xgboost', xgb.__version__)

xgboost 3.2.0


## 1. Load & filter

Keep only polls that matched a result (`has_result == 1`) and general-election cycles (even years 2018–2024).

In [2]:
d = pd.read_csv('polls_long_with_results.csv', low_memory=False)
d = d[d['has_result'] == 1].copy()

d['end_date']      = pd.to_datetime(d['end_date'], errors='coerce', format='mixed')
d['election_date'] = pd.to_datetime(d['election_date'], errors='coerce', format='mixed')
d['days_to_elec']  = (d['election_date'] - d['end_date']).dt.days

CYCLES = [2018, 2020, 2022, 2024]
d = d[d['year'].isin(CYCLES)].copy()

for c in ['pct','numeric_grade','pollscore','transparency_score','sample_size','days_to_elec']:
    d[c] = pd.to_numeric(d[c], errors='coerce')

print('poll rows:', len(d))
print('races:', d['race_id'].nunique())
print('by year (races):', d.groupby('year')['race_id'].nunique().to_dict())

poll rows: 16752
races: 687
by year (races): {2018: 227, 2020: 151, 2022: 176, 2024: 133}


## 1b. External features: partisan lean, incumbency, prior margin

Three signals polls don't capture, all leak-free (structural or strictly-prior data):

- **Partisan lean** — 538's `partisan-lean` dataset (state lean for Senate/Gov, district lean for House). A CPVI-style number; negative = R-leaning.
- **Incumbency** — `incumbent_party` per race from the `election-results` `races.csv`; we derive a per-candidate `is_incumbent` (this candidate's party holds the seat *and* they are running).
- **Prior same-office margin** — the most recent **previous** election's two-party margin for that exact seat, computed from results history (no current-cycle data, so no leakage).

All three are **signed toward the candidate's own party** so the model reads them consistently.

In [3]:
import io, re, requests

H = {'User-Agent': 'Mozilla/5.0 (research)'}
STATE_ABBR = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA','Colorado':'CO',
    'Connecticut':'CT','Delaware':'DE','District of Columbia':'DC','Florida':'FL','Georgia':'GA',
    'Hawaii':'HI','Idaho':'ID','Illinois':'IL','Indiana':'IN','Iowa':'IA','Kansas':'KS','Kentucky':'KY',
    'Louisiana':'LA','Maine':'ME','Maryland':'MD','Massachusetts':'MA','Michigan':'MI','Minnesota':'MN',
    'Mississippi':'MS','Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV','New Hampshire':'NH',
    'New Jersey':'NJ','New Mexico':'NM','New York':'NY','North Carolina':'NC','North Dakota':'ND',
    'Ohio':'OH','Oklahoma':'OK','Oregon':'OR','Pennsylvania':'PA','Rhode Island':'RI','South Carolina':'SC',
    'South Dakota':'SD','Tennessee':'TN','Texas':'TX','Utah':'UT','Vermont':'VT','Virginia':'VA',
    'Washington':'WA','West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY',
}

def npar(p):
    p = str(p).upper()
    return 'DEM' if p.startswith('DEM') else 'REP' if p.startswith('REP') else 'OTH'

def pdist(v):
    m = re.search(r'(\d+)', str(v))
    return str(int(m.group(1))) if m else ''

def _get(url):
    return pd.read_csv(io.StringIO(requests.get(url, timeout=90, headers=H).text), low_memory=False)

# --- 538 partisan lean (state + district) ---
LB = 'https://raw.githubusercontent.com/fivethirtyeight/data/master/partisan-lean/'
ls = _get(LB + 'fivethirtyeight_partisan_lean_STATES.csv'); ls.columns = ['sname', 'lean']
ls['state'] = ls['sname'].map(STATE_ABBR)
state_lean = dict(zip(ls['state'], ls['lean']))

ld = _get(LB + 'fivethirtyeight_partisan_lean_DISTRICTS.csv'); ld.columns = ['dcode', 'lean']
ld['state'] = ld['dcode'].str.split('-').str[0]
ld['district'] = ld['dcode'].str.split('-').str[1].map(lambda x: str(int(x)) if str(x).isdigit() else x)
dist_lean = {(r.state, r.district): r.lean for r in ld.itertuples()}

# --- incumbent_party per race (races.csv) ---
rc = _get('https://raw.githubusercontent.com/fivethirtyeight/election-results/main/races.csv')
def _off(o):
    o = str(o).lower()
    return 'Senate' if 'senate' in o else 'House' if 'house' in o else 'Governor' if 'governor' in o else None
rc['office'] = rc['office_name'].map(_off)
rc['state'] = rc['state_abbrev'].str.upper()
rc['district'] = ''
rc.loc[rc['office'] == 'House', 'district'] = rc.loc[rc['office'] == 'House', 'office_seat_name'].map(pdist)
inc_map = {(r.cycle, r.state, r.office, r.district): npar(r.incumbent_party)
           for r in rc[rc['office'].notna()].itertuples() if pd.notna(r.incumbent_party)}

# --- prior same-office two-party margin from results history (cached files in data/) ---
def _load_res(path, office):
    r = pd.read_csv(path, low_memory=False)
    r = r[r['stage'].astype(str).str.lower().str.contains('general', na=False)]
    r['office'] = office
    r['state'] = r['state_abbrev'].str.upper()
    r['district'] = '' if office != 'House' else r['office_seat_name'].map(pdist)
    r['p'] = r['ballot_party'].map(npar)      # 'party' col is null in these files; ballot_party holds DEM/REP
    r['pct'] = pd.to_numeric(r['percent'], errors='coerce')
    return r[['cycle', 'state', 'office', 'district', 'p', 'pct']]

allres = pd.concat([_load_res('data/res_senate.csv', 'Senate'),
                    _load_res('data/res_house.csv', 'House'),
                    _load_res('data/res_governor.csv', 'Governor')])
piv = (allres[allres['p'].isin(['DEM', 'REP'])]
       .groupby(['cycle', 'state', 'office', 'district', 'p'])['pct'].max().unstack('p'))
for col in ['DEM', 'REP']:
    if col not in piv.columns:
        piv[col] = np.nan
piv['margin'] = piv['DEM'].fillna(0) - piv['REP'].fillna(0)
margin_map = {idx: row.margin for idx, row in piv.iterrows()}

# Most recent same-office two-party margin strictly before `year` (leak-free)
def prior_margin(year, state, office, district):
    for back in range(2, 9, 2):
        v = margin_map.get((year - back, state, office, district))
        if v is not None and not (isinstance(v, float) and np.isnan(v)):
            return v
    return np.nan

# --- national environment: generic-ballot DEM-REP, last 30 days, per cycle (Internet Archive 538) ---
GB_URL = ('http://web.archive.org/web/2id_/'
          'https://projects.fivethirtyeight.com/polls/data/generic_ballot_polls_historical.csv')
gb = _get(GB_URL)
gb['end_date'] = pd.to_datetime(gb['end_date'], errors='coerce', format='mixed')
gb['cycle'] = pd.to_numeric(gb['cycle'], errors='coerce')
gb['dem'] = pd.to_numeric(gb['dem'], errors='coerce')
gb['rep'] = pd.to_numeric(gb['rep'], errors='coerce')
gb['elec'] = pd.to_datetime(gb['cycle'].astype('Int64').astype(str) + '-11-05', errors='coerce')
gb['dte'] = (gb['elec'] - gb['end_date']).dt.days
gb_late = gb[(gb['dte'] >= 0) & (gb['dte'] <= 30)]
natl_env = {int(c): (g['dem'].mean() - g['rep'].mean()) for c, g in gb_late.groupby('cycle')}  # + = D environment

print('state leans:', len(state_lean), '| district leans:', len(dist_lean),
      '| incumbent records:', len(inc_map), '| prior-margin races:', len(margin_map))
print('national environment (DEM-REP, last 30d):', {k: round(v, 1) for k, v in natl_env.items()})

state leans: 51 | district leans: 435 | incumbent records: 12242 | prior-margin races: 8600
national environment (DEM-REP, last 30d): {2018: np.float64(7.8), 2020: np.float64(7.5), 2022: np.float64(0.7), 2024: np.float64(0.1)}


## 1c. Macro / political-climate features

National per-cycle conditions: **presidential approval, inflation (YoY), real GDP growth, unemployment, and gas prices.** For each metric we compute campaign-year **trajectory stats** — election-eve level, mean, max, std (spread), and trend — not just a single snapshot (volatile inflation may matter more than its level, etc.).

These are national values (constant within a cycle), so on their own they can't rank candidates *within* a race. We add **`is_president_party`** (1 if the candidate's party holds the White House) as the interaction key — XGBoost can then learn effects like "low approval + in-party → headwind" without us hard-coding the sign.

Data: economic series from **FRED** (live download); approval + an election-eve fallback table are in `macro_features.py` so the cell runs even if FRED is unreachable. **With only 4 cycles these features are weak and must be used with heavy regularization** — see the re-tuned hyperparameters in section 10.

In [4]:
from macro_features import build_macro, PRES_PARTY

# Reads the static monthly macro CSV (data/macro_monthly.csv) produced once by fetch_macro.py.
# No network here — the data is fixed historical record. If the CSV is missing, this raises a
# clear error telling you to run `python fetch_macro.py` first.
macro = build_macro()
MACRO_FEATS = sorted({k for cyc in macro for k in macro[cyc]})
print('president party by cycle:', PRES_PARTY)
print(f'macro features per cycle: {len(MACRO_FEATS)}')
print('sample (2022):', {k: round(v, 2) for k, v in list(macro[2022].items())[:6]}, '...')

president party by cycle: {2018: 'REP', 2020: 'REP', 2022: 'DEM', 2024: 'DEM'}
macro features per cycle: 49
sample (2022): {'approval_eve': 42.0, 'approval_mean': 45.83, 'approval_max': 55.0, 'approval_min': 38.0, 'approval_std': 5.35, 'approval_trend': -0.63} ...


## 2. Poll weighting

Not all polls are equal. We weight each poll by:
- **recency** — closer to election day counts more (`1 / (1 + days/30)`),
- **sample size** — `sqrt(n)` (precision scales with root-n),
- **pollster quality** — 538 `numeric_grade`.

Used to compute a weighted polling average per candidate.

In [5]:
d['w'] = (1.0 / (1.0 + d['days_to_elec'].clip(lower=0) / 30.0)) \
         * np.sqrt(d['sample_size'].clip(lower=1)) \
         * (1.0 + d['numeric_grade'].fillna(0) / 3.0)

def wmean(values, weights):
    """Weighted mean, falling back to plain mean if weights are unusable."""
    w = weights.reindex(values.index).fillna(0)
    if w.sum() > 0 and values.notna().any():
        return np.average(values.fillna(values.mean()), weights=w)
    return values.mean()

def count_lead_changes(g):
    """How many times the front-runner changed in the polls over the race.
    Leader at each poll date = candidate with the highest *running mean* pct up to
    that date (running mean smooths out single outlier polls)."""
    g = g.dropna(subset=['end_date', 'pct']).sort_values('end_date')
    prev, changes = None, 0
    for dt in g['end_date'].drop_duplicates().sort_values():
        means = g[g['end_date'] <= dt].groupby('cand_key')['pct'].mean()
        if means.empty:
            continue
        leader = means.idxmax()
        if prev is not None and leader != prev:
            changes += 1
        prev = leader
    return changes

def margin_dynamics(g):
    """Per-candidate lead/deficit *trajectory* over the campaign.

    At each poll date, every candidate's running-mean pct is computed, and their
    margin = own running mean minus the best *other* candidate's running mean.
    Returns, per candidate, summary stats of that margin time series:
      avg_margin_over_time, margin_volatility (std), min_margin, margin_trend (slope vs time).
    """
    g = g.dropna(subset=['end_date', 'pct']).sort_values('end_date')
    dates = g['end_date'].drop_duplicates().sort_values()
    series = {}   # cand_key -> list[(elapsed_days, margin)]
    t0 = dates.min()
    for dt in dates:
        means = g[g['end_date'] <= dt].groupby('cand_key')['pct'].mean()
        if len(means) == 0:
            continue
        elapsed = (dt - t0).days
        for ck, val in means.items():
            others = means.drop(ck)
            best_other = others.max() if len(others) else 0.0
            series.setdefault(ck, []).append((elapsed, val - best_other))
    out = {}
    for ck, pts in series.items():
        m = np.array([p[1] for p in pts], dtype=float)
        x = np.array([p[0] for p in pts], dtype=float)
        trend = np.polyfit(x, m, 1)[0] if (len(m) >= 2 and np.ptp(x) > 0) else 0.0
        out[ck] = dict(avg_margin_over_time=float(np.mean(m)),
                       margin_volatility=float(np.std(m)) if len(m) > 1 else 0.0,
                       min_margin=float(np.min(m)),
                       margin_trend=float(trend))
    return out

# race-level lead-change counts; per-(race,candidate) margin trajectory stats
lead_change_map = d.groupby('race_id').apply(count_lead_changes, include_groups=False).to_dict()
margin_dyn_map = {rid: margin_dynamics(g) for rid, g in d.groupby('race_id')}

## 3. Collapse to one row per candidate per race

Aggregate every poll for a candidate into a feature vector. Then add **race-relative** features (lead over best opponent, share of the polled field) — these matter more than raw % because elections are relative.

In [6]:
d['district'] = d['district'].fillna('').astype(str).replace('nan', '')

# --- pollster house effect (TRAIN cycles only, to avoid leakage) ---
mar = (d[d['party_std'].isin(['DEM', 'REP'])]
       .pivot_table(index=['race_id', 'poll_id', 'pollster', 'year'],
                    columns='party_std', values='pct', aggfunc='max').reset_index())
for col in ['DEM', 'REP']:
    if col not in mar.columns:
        mar[col] = np.nan
mar['m'] = mar['DEM'] - mar['REP']
tm = mar[mar['year'] <= 2022].copy()
tm['dev'] = tm['m'] - tm.groupby('race_id')['m'].transform('mean')
house = tm.groupby('pollster')['dev'].mean().to_dict()

rows = []
for race_id, g in d.groupby('race_id'):
    yr = int(g['year'].iloc[0]); st = g['state'].iloc[0]
    of = g['office'].iloc[0];    di = str(g['district'].iloc[0])
    dyn = margin_dyn_map.get(race_id, {})
    for ck, gc in g.groupby('cand_key'):
        gc = gc.sort_values('end_date')
        last30 = gc[gc['days_to_elec'] <= 30]
        party = gc['party_std'].iloc[0]
        sign = 1 if party == 'DEM' else -1 if party == 'REP' else 0

        lean = dist_lean.get((st, di)) if of == 'House' else state_lean.get(st)
        incp = inc_map.get((yr, st, of, di))
        pm   = prior_margin(yr, st, of, di)

        last60 = gc[gc['days_to_elec'] <= 60].dropna(subset=['pct', 'days_to_elec'])
        slope = np.nan
        if len(last60) >= 3:
            x = -last60['days_to_elec'].values.astype(float)
            y = last60['pct'].values.astype(float)
            if np.ptp(x) > 0:
                slope = np.polyfit(x, y, 1)[0]

        adj = gc['pct'] - gc['pollster'].map(lambda p: sign * house.get(p, 0.0))
        md = dyn.get(ck, {})

        rows.append(dict(
            race_id=race_id, year=yr, state=st, office=of, district=di,
            cand_key=ck, candidate=gc['candidate'].iloc[0], party=party,
            won=int(gc['won'].iloc[0]),
            poll_avg=gc['pct'].mean(),
            poll_wavg=wmean(gc['pct'], gc['w']),
            poll_last=gc['pct'].iloc[-1],
            poll_last30=(last30['pct'].mean() if len(last30) else gc['pct'].mean()),
            poll_std=gc['pct'].std(),
            n_polls=len(gc),
            n_polls_over50=int((gc['pct'] > 50).sum()),
            avg_grade=gc['numeric_grade'].mean(),
            avg_pollscore=gc['pollscore'].mean(),
            avg_sample=gc['sample_size'].mean(),
            min_days=gc['days_to_elec'].min(),
            lean_cand=(sign * lean if lean is not None else np.nan),
            prior_margin_cand=(sign * pm if not (isinstance(pm, float) and np.isnan(pm)) else np.nan),
            is_incumbent=(1 if incp and incp == party else 0),
            is_inc_party_race=(1 if incp in ('DEM', 'REP') else 0),
            natl_env_cand=(sign * natl_env.get(yr, np.nan)),
            poll_momentum=slope,
            poll_wavg_adj=wmean(adj, gc['w']),
            n_lead_changes=lead_change_map.get(race_id, 0),
            lead_changed=int(lead_change_map.get(race_id, 0) > 0),
            avg_margin_over_time=md.get('avg_margin_over_time', np.nan),
            margin_volatility=md.get('margin_volatility', np.nan),
            min_margin=md.get('min_margin', np.nan),
            margin_trend=md.get('margin_trend', np.nan),
            # macro / political climate: national per-cycle values + the interaction key.
            is_president_party=int(party == PRES_PARTY.get(yr)),
            **macro[yr],
        ))
cand = pd.DataFrame(rows)

# race-relative features
cand['field_best'] = cand.groupby('race_id')['poll_wavg'].transform(
    lambda s: s.nlargest(2).min() if len(s) > 1 else s.max())
cand['poll_lead']  = cand['poll_wavg'] - cand['field_best']
cand['poll_share'] = cand['poll_wavg'] / cand.groupby('race_id')['poll_wavg'].transform('sum')
cand['n_cands']    = cand.groupby('race_id')['cand_key'].transform('count')
cand['race_total_polls']  = cand.groupby('race_id')['n_polls'].transform('sum')
cand['frac_polls_over50'] = cand['n_polls_over50'] / cand['n_polls']
cand['is_dem']     = (cand['party'] == 'DEM').astype(int)
cand['is_rep']     = (cand['party'] == 'REP').astype(int)
cand['is_senate']  = (cand['office'] == 'Senate').astype(int)
cand['is_gov']     = (cand['office'] == 'Governor').astype(int)

# --- gap cluster ---
dem_w = cand[cand['party'] == 'DEM'].groupby('race_id')['poll_wavg'].max()
rep_w = cand[cand['party'] == 'REP'].groupby('race_id')['poll_wavg'].max()
twoparty = (dem_w - rep_w)
cand['twoparty_margin_cand'] = (cand['race_id'].map(twoparty)
                                * cand['party'].map({'DEM': 1, 'REP': -1}).fillna(0))
cand['abs_gap']    = cand['race_id'].map(twoparty.abs())
cand['tossup']     = (cand['abs_gap'] < 3).astype(int)
cand['undecided']  = 100 - cand.groupby('race_id')['poll_wavg'].transform('sum')
cand['gap_x_recency'] = cand['poll_lead'] * (1.0 / (1.0 + cand['min_days'].clip(lower=0) / 30.0))

# dynamically pick a couple of macro columns that exist, for the preview
_macro_preview = [c for c in ['approval_yr_eve', 'inflation_yr_max', 'gas_yr_max'] if c in cand.columns]
print('candidate-races:', len(cand), '| win rate:', round(cand['won'].mean(), 3))
print('is_president_party rate:', round(cand['is_president_party'].mean(), 3))
print('macro columns present:', sum(c in cand.columns for c in MACRO_FEATS), 'of', len(MACRO_FEATS))
cand[['year','state','office','candidate','party','poll_wavg','poll_lead',
      'is_president_party'] + _macro_preview + ['won']].head(8)

candidate-races: 1859 | win rate: 0.371
is_president_party rate: 0.365
macro columns present: 49 of 49


,year,state,office,candidate,party,poll_wavg,poll_lead,is_president_party,won
0,2018,AK,Governor,Mark Begich,DEM,35.388797,0.000000,0,0
1,2018,AK,Governor,Mike Dunleavy,REP,44.673941,9.285143,1,1
2,2018,AK,Governor,Billy Toien,OTH,3.300000,-32.088797,0,0
3,2018,AK,Governor,Bill Walker,OTH,23.189473,-12.199325,0,0
4,2018,AK,House,Alyse S. Galvin,DEM,45.252545,0.000000,0,0
5,2018,AK,House,Don Young,REP,48.463695,3.211150,1,1
6,2018,AL,Governor,Kay Ivey,REP,53.483009,21.531468,1,1
7,2018,AL,Governor,Walter Maddox,DEM,31.951541,0.000000,0,0


## 4. Train / test split by year

Train on 2018/2020/2022, test on 2024. **Never** include `vote_pct` / `race_winning_pct` — those are outcomes.

In [7]:
FEATURES = [
    'poll_avg','poll_wavg','poll_last','poll_last30','poll_std','n_polls',
    'n_polls_over50','frac_polls_over50','race_total_polls',
    'avg_grade','avg_pollscore','avg_sample','min_days',
    'poll_lead','poll_share','n_cands',
    'is_dem','is_rep','is_senate','is_gov',
    # fundamentals
    'lean_cand','prior_margin_cand','is_incumbent','is_inc_party_race',
    # gap cluster
    'twoparty_margin_cand','abs_gap','tossup','undecided','gap_x_recency',
    # environment / momentum / house-adjusted poll
    'natl_env_cand','poll_momentum','poll_wavg_adj',
    # lead dynamics over time
    'n_lead_changes','lead_changed','avg_margin_over_time','margin_volatility','min_margin','margin_trend',
    # macro / political climate (national per-cycle) + interaction key
    'is_president_party',
] + MACRO_FEATS

train = cand[cand['year'] <= 2022]
test  = cand[cand['year'] == 2024]
X_train, y_train = train[FEATURES], train['won']
X_test,  y_test  = test[FEATURES],  test['won']
print('train:', X_train.shape, '| test:', X_test.shape, '| total features:', len(FEATURES))
print('train win rate:', round(y_train.mean(), 3), '| test win rate:', round(y_test.mean(), 3))

train: (1515, 88) | test: (344, 88) | total features: 88
train win rate: 0.367 | test win rate: 0.387


In [8]:
# ---- feature builder + leave-one-cycle-out folds (defined here so tuning below can use them) ----
# build_features rebuilds the candidate table for a given set of TRAIN cycles, recomputing the
# pollster house-effect from train cycles only (leak-free). Used by the CV folds and the grid search.
def build_features(train_years):
    mar = (d[d['party_std'].isin(['DEM', 'REP'])]
           .pivot_table(index=['race_id', 'poll_id', 'pollster', 'year'],
                        columns='party_std', values='pct', aggfunc='max').reset_index())
    for col in ['DEM', 'REP']:
        if col not in mar.columns:
            mar[col] = np.nan
    mar['m'] = mar['DEM'] - mar['REP']
    tm = mar[mar['year'].isin(train_years)].copy()
    tm['dev'] = tm['m'] - tm.groupby('race_id')['m'].transform('mean')
    house = tm.groupby('pollster')['dev'].mean().to_dict()

    rows = []
    for race_id, g in d.groupby('race_id'):
        yr = int(g['year'].iloc[0]); st = g['state'].iloc[0]
        of = g['office'].iloc[0];    di = str(g['district'].iloc[0])
        dyn = margin_dyn_map.get(race_id, {})
        for ck, gc in g.groupby('cand_key'):
            gc = gc.sort_values('end_date')
            last30 = gc[gc['days_to_elec'] <= 30]
            party = gc['party_std'].iloc[0]
            sign = 1 if party == 'DEM' else -1 if party == 'REP' else 0
            lean = dist_lean.get((st, di)) if of == 'House' else state_lean.get(st)
            incp = inc_map.get((yr, st, of, di)); pm = prior_margin(yr, st, of, di)
            last60 = gc[gc['days_to_elec'] <= 60].dropna(subset=['pct', 'days_to_elec'])
            slope = np.nan
            if len(last60) >= 3:
                x = -last60['days_to_elec'].values.astype(float); y = last60['pct'].values.astype(float)
                if np.ptp(x) > 0:
                    slope = np.polyfit(x, y, 1)[0]
            adj = gc['pct'] - gc['pollster'].map(lambda p: sign * house.get(p, 0.0))
            md = dyn.get(ck, {})
            rows.append(dict(
                race_id=race_id, year=yr, office=of, cand_key=ck, party=party, won=int(gc['won'].iloc[0]),
                poll_avg=gc['pct'].mean(), poll_wavg=wmean(gc['pct'], gc['w']), poll_last=gc['pct'].iloc[-1],
                poll_last30=(last30['pct'].mean() if len(last30) else gc['pct'].mean()), poll_std=gc['pct'].std(),
                n_polls=len(gc), n_polls_over50=int((gc['pct'] > 50).sum()), avg_grade=gc['numeric_grade'].mean(),
                avg_pollscore=gc['pollscore'].mean(), avg_sample=gc['sample_size'].mean(), min_days=gc['days_to_elec'].min(),
                lean_cand=(sign * lean if lean is not None else np.nan),
                prior_margin_cand=(sign * pm if not (isinstance(pm, float) and np.isnan(pm)) else np.nan),
                is_incumbent=(1 if incp and incp == party else 0), is_inc_party_race=(1 if incp in ('DEM', 'REP') else 0),
                natl_env_cand=(sign * natl_env.get(yr, np.nan)), poll_momentum=slope, poll_wavg_adj=wmean(adj, gc['w']),
                n_lead_changes=lead_change_map.get(race_id, 0), lead_changed=int(lead_change_map.get(race_id, 0) > 0),
                avg_margin_over_time=md.get('avg_margin_over_time', np.nan),
                margin_volatility=md.get('margin_volatility', np.nan),
                min_margin=md.get('min_margin', np.nan), margin_trend=md.get('margin_trend', np.nan),
                is_president_party=int(party == PRES_PARTY.get(yr)), **macro[yr],
            ))
    c = pd.DataFrame(rows)
    c['field_best'] = c.groupby('race_id')['poll_wavg'].transform(lambda s: s.nlargest(2).min() if len(s) > 1 else s.max())
    c['poll_lead']  = c['poll_wavg'] - c['field_best']
    c['poll_share'] = c['poll_wavg'] / c.groupby('race_id')['poll_wavg'].transform('sum')
    c['n_cands']    = c.groupby('race_id')['cand_key'].transform('count')
    c['race_total_polls']  = c.groupby('race_id')['n_polls'].transform('sum')
    c['frac_polls_over50'] = c['n_polls_over50'] / c['n_polls']
    c['is_dem'] = (c['party'] == 'DEM').astype(int); c['is_rep'] = (c['party'] == 'REP').astype(int)
    c['is_senate'] = (c['office'] == 'Senate').astype(int); c['is_gov'] = (c['office'] == 'Governor').astype(int)
    dem_w = c[c['party'] == 'DEM'].groupby('race_id')['poll_wavg'].max()
    rep_w = c[c['party'] == 'REP'].groupby('race_id')['poll_wavg'].max(); tp = dem_w - rep_w
    c['twoparty_margin_cand'] = c['race_id'].map(tp) * c['party'].map({'DEM': 1, 'REP': -1}).fillna(0)
    c['abs_gap'] = c['race_id'].map(tp.abs()); c['tossup'] = (c['abs_gap'] < 3).astype(int)
    c['undecided'] = 100 - c.groupby('race_id')['poll_wavg'].transform('sum')
    c['gap_x_recency'] = c['poll_lead'] * (1.0 / (1.0 + c['min_days'].clip(lower=0) / 30.0))
    return c

CYCLES_TRAIN = CYCLES
FOLDS = {ty: build_features([y for y in CYCLES if y != ty]) for ty in CYCLES}  # leave-one-cycle-out

def cv_auc_brier(F, params):
    aucs, briers = [], []
    for ty in CYCLES:
        c = FOLDS[ty]; tr = c[c['year'] != ty]; te = c[c['year'] == ty]
        m = xgb.XGBClassifier(eval_metric='logloss', random_state=42, **params)
        m.fit(tr[F], tr['won']); p = m.predict_proba(te[F])[:, 1]
        aucs.append(roc_auc_score(te['won'], p)); briers.append(brier_score_loss(te['won'], p))
    return float(np.mean(aucs)), float(np.mean(briers))

print('folds built:', list(FOLDS), '| feature builder ready')

folds built: [2018, 2020, 2022, 2024] | feature builder ready


## 5. Tune hyperparameters (leave-one-cycle-out grid search) & train

**Workflow rule:** the hyperparameters are *searched here every run*, not hardcoded — because params tuned for one feature set can make a different feature set look worse (adding the macro features with stale params showed a regression that re-tuning erased). The grid is scored by leave-one-cycle-out CV AUC. Heavy-regularization options (depth-1 stumps, high `reg_lambda`, low `colsample`) let XGBoost ignore non-predictive features rather than overfitting the 4 cycles.

In [9]:
import itertools, random

# Leave-one-cycle-out grid search. FOLDS + cv_auc_brier are defined in the cell just above.
GRID = dict(max_depth=[1, 2], learning_rate=[0.02, 0.03, 0.05],
            n_estimators=[150, 300], min_child_weight=[8, 15, 30],
            subsample=[0.6, 0.8, 1.0], colsample_bytree=[0.4, 0.6, 1.0],
            reg_lambda=[1, 5, 20], reg_alpha=[0, 1])
keys = list(GRID)
combos = list(itertools.product(*[GRID[k] for k in keys]))
random.seed(0); random.shuffle(combos); combos = combos[:150]   # sample 150 configs for speed

best = None
for combo in combos:
    p = dict(zip(keys, combo))
    a, b = cv_auc_brier(FEATURES, p)
    if best is None or a > best[0]:
        best = (a, b, p)

XGB_PARAMS = dict(best[2], eval_metric='logloss', random_state=42)
print(f'best CV  AUC {best[0]:.4f}  Brier {best[1]:.4f}')
print('XGB_PARAMS =', {k: v for k, v in XGB_PARAMS.items() if k not in ('eval_metric', 'random_state')})

# fit the single-split model (train<=2022, test 2024) with the tuned params
model = xgb.XGBClassifier(**XGB_PARAMS)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
pred  = (proba > 0.5).astype(int)

best CV  AUC 0.9645  Brier 0.0740
XGB_PARAMS = {'max_depth': 1, 'learning_rate': 0.05, 'n_estimators': 150, 'min_child_weight': 8, 'subsample': 0.6, 'colsample_bytree': 0.4, 'reg_lambda': 20, 'reg_alpha': 0}


## 6. Evaluate (candidate level)

In [10]:
print('=== candidate-level (test = 2024) ===')
print('AUC      ', round(roc_auc_score(y_test, proba), 3))
print('LogLoss  ', round(log_loss(y_test, proba), 3))
print('Brier    ', round(brier_score_loss(y_test, proba), 3), ' (lower = better calibrated)')
print('Accuracy ', round(accuracy_score(y_test, pred), 3))
print('\nConfusion matrix [rows=true, cols=pred]:')
print(confusion_matrix(y_test, pred))

=== candidate-level (test = 2024) ===
AUC       0.969
LogLoss   0.231
Brier     0.071  (lower = better calibrated)
Accuracy  0.892

Confusion matrix [rows=true, cols=pred]:
[[191  20]
 [ 17 116]]


## 7. Evaluate (race level) + baseline

Real question: per race, did we pick the right winner? Compare the model (highest predicted prob) to the naive baseline (highest polling average).

In [11]:
te = test.copy()
te['proba'] = proba

model_pick = te.loc[te.groupby('race_id')['proba'].idxmax()]
base_pick  = te.loc[te.groupby('race_id')['poll_wavg'].idxmax()]   # naive: highest weighted poll

print('races in test:', te['race_id'].nunique())
print('model  winner accuracy:', round(model_pick['won'].mean(), 3))
print('baseline (top poll)   :', round(base_pick['won'].mean(), 3))

# by office — Senate/Gov are statewide & well-polled (polls ~perfect); House is the hard part
print('\nby office  (model vs baseline, n races):')
for of in ['Senate', 'Governor', 'House']:
    mp = model_pick[model_pick['office'] == of]
    bp = base_pick[base_pick['office'] == of]
    if len(mp):
        print(f'  {of:9}  model {mp["won"].mean():.3f}   baseline {bp["won"].mean():.3f}   (n={len(mp)})')

print('\nRaces the model got WRONG:')
wrong = model_pick[model_pick['won'] == 0]
wrong[['year','state','office','district','candidate','party','poll_wavg',
       'poll_lead','lean_cand','prior_margin_cand','is_incumbent','proba']]

races in test: 133
model  winner accuracy: 0.872
baseline (top poll)   : 0.88

by office  (model vs baseline, n races):
  Senate     model 0.938   baseline 0.969   (n=32)
  Governor   model 1.000   baseline 1.000   (n=11)
  House      model 0.833   baseline 0.833   (n=90)

Races the model got WRONG:


,year,state,office,district,candidate,party,poll_wavg,poll_lead,lean_cand,prior_margin_cand,is_incumbent,proba
1522,2024,AZ,House,1.0,Amish Shah,DEM,48.000000,0.000000,NaN,NaN,0,0.568519
1524,2024,AZ,House,2.0,Jonathan Nez,DEM,42.000000,0.000000,NaN,NaN,0,0.313881
1537,2024,CA,House,22.0,Rudy Salas,DEM,45.334481,0.674997,NaN,NaN,0,0.535361
1542,2024,CA,House,41.0,Will Rollins,DEM,44.320121,1.783927,NaN,NaN,0,0.784714
1545,2024,CA,House,47.0,Scott Baugh,REP,44.227667,2.654014,NaN,NaN,0,0.764255
1558,2024,CA,House,9.0,Kevin Lincoln,REP,44.000000,4.000000,NaN,NaN,0,0.711790
1561,2024,CO,House,3.0,Adam Frisch,DEM,48.017227,0.000000,NaN,NaN,0,0.774852
1593,2024,IA,House,1.0,Christina Bohannan,DEM,47.410709,2.046885,NaN,NaN,0,0.652404
1595,2024,IA,House,3.0,Lanon Baccam,DEM,45.328913,2.759508,NaN,NaN,0,0.853997
1630,2024,MI,House,8.0,Paul Junge,REP,42.252054,1.218715,NaN,NaN,0,0.558961


## 8. Feature importance

In [12]:
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
ax = imp.plot.barh(figsize=(7, 5), title='XGBoost feature importance')
ax.set_xlabel('gain importance')
plt.tight_layout(); plt.show()
imp.sort_values(ascending=False).round(3)

C:\Users\pjmer\AppData\Local\Temp\ipykernel_17940\3632646110.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


gap_x_recency           0.196
poll_lead               0.173
poll_wavg_adj           0.117
poll_share              0.096
twoparty_margin_cand    0.080
                        ...  
inflation_min           0.000
inflation_eve           0.000
gas_trend               0.000
gas_std                 0.000
gas_min                 0.000
Length: 88, dtype: float32

## 9. Calibration curve

Are predicted probabilities honest? A well-calibrated model: candidates given ~70% win ~70% of the time.

In [13]:
frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=8, strategy='quantile')
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], '--', color='gray', label='perfect')
plt.plot(mean_pred, frac_pos, 'o-', label='model')
plt.xlabel('predicted win probability'); plt.ylabel('actual win frequency')
plt.title('Calibration (2024 test)'); plt.legend(); plt.tight_layout(); plt.show()

C:\Users\pjmer\AppData\Local\Temp\ipykernel_17940\4115570372.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('Calibration (2024 test)'); plt.legend(); plt.tight_layout(); plt.show()


## 10. Leave-one-cycle-out cross-validation

A single 2024 test set is one data point — and 2024 turned out to be the model's *best* cycle, so it flatters the model. Honest validation rotates the test cycle: train on three cycles, test on the held-out fourth, repeat for all four, then average.

This also makes the **pollster house-effect** fully leak-free — it's recomputed from each fold's training cycles only (the single-split version above fit it on 2018–2022, which would leak if 2018 were the test fold).

In [14]:
def make_model():
    return xgb.XGBClassifier(**XGB_PARAMS)   # tuned in section 5

# FOLDS were built once in section 4; reuse them (each is leave-one-cycle-out, leak-free).
cv_rows = []
for test_y in CYCLES:
    c = FOLDS[test_y]
    tr = c[c['year'] != test_y]; te = c[c['year'] == test_y].copy()
    m = make_model(); m.fit(tr[FEATURES], tr['won'])
    te['p'] = m.predict_proba(te[FEATURES])[:, 1]
    pick = te.loc[te.groupby('race_id')['p'].idxmax()]
    base = te.loc[te.groupby('race_id')['poll_wavg'].idxmax()]
    cv_rows.append(dict(
        test_cycle=test_y, n_cand=len(te), n_races=te['race_id'].nunique(),
        AUC=roc_auc_score(te['won'], te['p']),
        Brier=brier_score_loss(te['won'], te['p']),
        LogLoss=log_loss(te['won'], te['p']),
        race_acc=pick['won'].mean(), baseline_acc=base['won'].mean(),
    ))

cv = pd.DataFrame(cv_rows).set_index('test_cycle')
print('=== Leave-one-cycle-out CV (tuned model, with macro features) ===')
print(cv.round(3).to_string())
print('\nMEAN over cycles:')
print(cv[['AUC', 'Brier', 'LogLoss', 'race_acc', 'baseline_acc']].mean().round(3).to_string())

=== Leave-one-cycle-out CV (tuned model, with macro features) ===
            n_cand  n_races    AUC  Brier  LogLoss  race_acc  baseline_acc
test_cycle                                                                
2018           639      227  0.965  0.068    0.233     0.868         0.890
2020           394      151  0.956  0.088    0.279     0.828         0.828
2022           482      176  0.968  0.069    0.233     0.881         0.892
2024           344      133  0.969  0.071    0.231     0.872         0.880

MEAN over cycles:
AUC             0.964
Brier           0.074
LogLoss         0.244
race_acc        0.862
baseline_acc    0.872


## 11. Honest benchmark: does the model beat a pure poll signal?

The single biggest question for a project like this: **does the ML model actually beat just using the polls?** Here we build a trivial baseline — convert each candidate's weighted poll average into a win probability via a softmax within the race — and compare it to the XGBoost model under the same leave-one-cycle-out CV.

We also sweep a **blend** `α·model + (1−α)·poll` to see whether mixing helps (α=1 is pure model, α=0 is pure poll).

In [15]:
def poll_softmax(g, temp=3.0):
    """Turn weighted poll averages into within-race win probabilities (smooth 'leader' signal)."""
    v = g['poll_wavg'].fillna(g['poll_wavg'].mean()).values
    e = np.exp((v - np.nanmax(v)) / temp)
    return pd.Series(e / e.sum(), index=g.index)

def race_winner_acc(te, col):
    pick = te.loc[te.groupby('race_id')[col].idxmax()]
    return pick['won'].mean()

# build per-fold predictions once (model prob + poll prob)
fold_pred = {}
for test_y in CYCLES:
    c = build_features([y for y in CYCLES if y != test_y])
    tr = c[c['year'] != test_y]
    te = c[c['year'] == test_y].copy()
    m = make_model(); m.fit(tr[FEATURES], tr['won'])
    te['p_model'] = m.predict_proba(te[FEATURES])[:, 1]
    te['p_poll']  = te.groupby('race_id', group_keys=False).apply(poll_softmax)
    fold_pred[test_y] = te

def cv_metrics(col):
    a = np.mean([roc_auc_score(fold_pred[ty]['won'], fold_pred[ty][col]) for ty in CYCLES])
    b = np.mean([brier_score_loss(fold_pred[ty]['won'], fold_pred[ty][col]) for ty in CYCLES])
    r = np.mean([race_winner_acc(fold_pred[ty], col) for ty in CYCLES])
    return a, b, r

am, bm, rm = cv_metrics('p_model')
ap, bp, rp = cv_metrics('p_poll')
print('=== CV: XGBoost model vs pure poll signal ===')
print(f'  XGBoost model : AUC {am:.3f}  Brier {bm:.3f}  race-acc {rm:.3f}')
print(f'  poll softmax  : AUC {ap:.3f}  Brier {bp:.3f}  race-acc {rp:.3f}')

print('\n=== Blend sweep  (alpha=1 pure model, 0 pure poll) ===')
print('alpha   AUC    Brier  race-acc')
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    aucs, briers, accs = [], [], []
    for ty in CYCLES:
        te = fold_pred[ty]
        blend = a * te['p_model'] + (1 - a) * te['p_poll']
        aucs.append(roc_auc_score(te['won'], blend))
        briers.append(brier_score_loss(te['won'], blend))
        pick = te.loc[blend.groupby(te['race_id']).idxmax()]
        accs.append(pick['won'].mean())
    print(f'{a:.2f}   {np.mean(aucs):.3f}  {np.mean(briers):.3f}  {np.mean(accs):.3f}')

print('\nTakeaway: the poll softmax matches/beats the model on every metric, and the blend')
print('is best at low alpha -> for win/lose, polls are the ceiling; the model adds little.')

=== CV: XGBoost model vs pure poll signal ===
  XGBoost model : AUC 0.964  Brier 0.074  race-acc 0.862
  poll softmax  : AUC 0.965  Brier 0.071  race-acc 0.872

=== Blend sweep  (alpha=1 pure model, 0 pure poll) ===
alpha   AUC    Brier  race-acc
0.00   0.965  0.071  0.872
0.25   0.967  0.070  0.870
0.50   0.967  0.071  0.865
0.75   0.966  0.072  0.864
1.00   0.964  0.074  0.862

Takeaway: the poll softmax matches/beats the model on every metric, and the blend
is best at low alpha -> for win/lose, polls are the ceiling; the model adds little.


## Notes / what's in here

**Feature families (all leak-free):** polls (weighted avg, last-30d, n_polls, polls-over-50%, std); race-relative gap (`poll_lead`, `twoparty_margin_cand`, `abs_gap`, `gap_x_recency` — a top feature); lead dynamics over time (`avg_margin_over_time` — top-4; `min_margin`; trend/volatility/lead-changes); fundamentals (`lean_cand`, `prior_margin_cand`, `is_incumbent`); environment/quality (`natl_env_cand`, `poll_momentum`, `poll_wavg_adj`).

**Hyperparameters** are CV-tuned (section 10): `max_depth=2, lr=0.03, n_estimators=200`.

**Cross-validated performance (leave-one-cycle-out):** AUC ≈ 0.96, Brier ≈ 0.08, race-winner accuracy ≈ 0.85.

### The headline honest finding (section 11)
**A trivial poll-only baseline beats this model.** Converting the weighted poll average into a within-race probability (softmax) scores AUC ≈ 0.965 / Brier ≈ 0.071 / race-acc ≈ 0.872 — better than the 38-feature XGBoost (≈0.959 / 0.080 / 0.846) on *every* CV metric. A blend `α·model + (1−α)·poll` is best at **α ≈ 0** (pure poll). 

Interpretation: for U.S. Senate/House/Governor **win-or-lose** prediction, the late polling average is so dominant that engineered features and gradient boosting add essentially nothing — and slightly hurt, by occasionally overriding a correct narrow poll leader. This is a real, well-known property of these races, not a bug in the pipeline. The model is still a perfectly good, well-calibrated probability model; it just isn't *better than the polls it's built on*.

### Where a model could actually add value (next steps)
- **Predict margin / overperformance vs polls**, not win/lose — a harder target where features have room to help.
- **House-specific model** with a time-varying district PVI (the one place polls are thin).
- **Probability calibration** (isotonic/Platt) if you only care about Brier.
- **Don't** add `vote_pct` / `race_winning_pct` — they're the answer.